[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishansurdi/SDK-Sovereign/blob/master/notebooks/02_train_lead.ipynb)

# 02 — Train Lead Adapter (GRPO)

**Purpose:** Fine-tune the Lead LoRA adapter using GRPO on rollout data collected from the live env. Upload trained adapter to HF Hub.

**Runtime:** Colab T4 · ~3h

> **Before running:** set `HF_TOKEN` + `WANDB_API_KEY` in Colab Secrets. Runtime → T4 GPU.

In [ ]:
# Cell 1 — Install + clone repo
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15.0" peft accelerate bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub datasets
!git clone https://github.com/ishansurdi/SDK-Sovereign.git sdk_sovereign_pkg 2>/dev/null || (cd sdk_sovereign_pkg && git pull)
!pip install -q -e sdk_sovereign_pkg/
import sys; sys.path.insert(0, 'sdk_sovereign_pkg')
print('✓ Ready')

In [ ]:
# Cell 2 — Config
HF_USER  = 'ishansurdi'
ENV_URL  = 'https://ishansurdi-sdk-sovereign.hf.space'
N_ROLLOUT_EPISODES = 80
WANDB_PROJECT = 'sdk-sovereign'
WANDB_ENTITY = 'OpenEnvR2'
FORCE_ROLLOUT_REBUILD = False
FORCE_TRAIN_REBUILD = False

In [ ]:
# Cell 3 — Load model + agents (re-run if kernel is fresh)
from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from google.colab import userdata
from huggingface_hub import login
import wandb, os

login(token=userdata.get('HF_TOKEN'))
wandb_key = userdata.get('WANDB_API_KEY')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    wandb.login(key=wandb_key)

model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print('✓ Model + agents loaded')

In [ ]:
# Cell A — Collect rollouts (resume-safe)
import json
from pathlib import Path
from client import SDKSovereignEnv
from models import SDKAction

ROLLOUT_LEAD_PATH = Path('rollout_lead.jsonl')
ROLLOUT_AUDITOR_PATH = Path('rollout_auditor.jsonl')

def _load_jsonl(path: Path):
    if not path.exists():
        return []
    lines = [ln for ln in path.read_text().splitlines() if ln.strip()]
    return [json.loads(ln) for ln in lines]

def _write_jsonl(path: Path, rows):
    path.write_text('\n'.join(json.dumps(r) for r in rows))

rollout_data = {
    'lead': _load_jsonl(ROLLOUT_LEAD_PATH),
    'auditor': _load_jsonl(ROLLOUT_AUDITOR_PATH),
}

if rollout_data['lead'] and rollout_data['auditor'] and not FORCE_ROLLOUT_REBUILD:
    lead_has_ep = all('episode' in r for r in rollout_data['lead'])
    auditor_has_ep = all('episode' in r for r in rollout_data['auditor'])
    if lead_has_ep and auditor_has_ep:
        done_lead = max((r['episode'] for r in rollout_data['lead']), default=-1) + 1
        done_auditor = max((r['episode'] for r in rollout_data['auditor']), default=-1) + 1
        start_ep = min(done_lead, done_auditor)
    else:
        start_ep = N_ROLLOUT_EPISODES
        print('Existing rollout files found without episode metadata; reusing as-is.')
else:
    if FORCE_ROLLOUT_REBUILD:
        rollout_data = {'auditor': [], 'lead': []}
    start_ep = 0

for ep in range(start_ep, N_ROLLOUT_EPISODES):
    with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
        obs = env.reset()
        per_role_buffer = []
        while not obs.done and obs.turn_index < 7:
            agent = agents[obs.current_role]
            agent.model.set_adapter(agent.adapter_name)
            prompt = agent._build_prompt(obs)
            completion = agent._generate(prompt)
            action = agent._parse_action(completion)
            new_obs = env.step(action)
            per_role_buffer.append({
                'episode': ep,
                'role': obs.current_role,
                'prompt': prompt,
                'completion': completion,
                'step_reward': new_obs.reward,
            })
            obs = new_obs

        for entry in per_role_buffer:
            rollout_data[entry['role']].append({
                'episode': entry['episode'],
                'prompt': entry['prompt'],
                'reward': entry['step_reward'],
            })

    # Flush each episode so Colab interruptions don't lose progress.
    _write_jsonl(ROLLOUT_LEAD_PATH, rollout_data['lead'])
    _write_jsonl(ROLLOUT_AUDITOR_PATH, rollout_data['auditor'])
    if ep % 5 == 0:
        print(f'  rollout {ep}/{N_ROLLOUT_EPISODES}')

print(f'Lead samples:    {len(rollout_data["lead"])}')
print(f'Auditor samples: {len(rollout_data["auditor"])}')
print('✓ Rollout data saved')

In [ ]:
# Cell B — GRPO train Lead adapter (resume-safe)
import subprocess, sys
from pathlib import Path
try:
    from trl import GRPOTrainer, GRPOConfig
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'trl>=0.15.0'])
    from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name='lead-grpo-round1')

lead_prompts = [r['prompt'] for r in rollout_data['lead']]
ds_lead = Dataset.from_dict({'prompt': lead_prompts})

def lead_reward_fn(completions, **kwargs):
    rewards = []
    for completion in completions:
        action = agents['lead']._parse_action(completion)
        if action.action_type == 'submit_patch':
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                env.step(SDKAction(role='auditor', action_type='pass'))
                env.step(SDKAction(role='lead', action_type='propose_replacement',
                                   proposed_sdk='razorpay'))
                env.step(SDKAction(role='auditor', action_type='approve'))
                final = env.step(action)
                rewards.append(float(final.reward))
        elif action.action_type == 'propose_replacement':
            rewards.append(1.0)
        elif action.action_type == 'request_hint':
            rewards.append(0.3)
        else:
            rewards.append(-0.5)  # pass
    return rewards

config = GRPOConfig(
    output_dir='checkpoints/lead',
    num_generations=4,
    max_completion_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=2,
    save_steps=50,
    report_to='wandb',
)

lead_artifact_dir = Path('artifacts/lead_adapter')
already_trained = (lead_artifact_dir / 'adapter_config.json').exists()

if already_trained and not FORCE_TRAIN_REBUILD:
    print('✓ Existing lead adapter artifact found; skipping training.')
else:
    ckpt_dir = Path('checkpoints/lead')
    checkpoints = sorted(
        [p for p in ckpt_dir.glob('checkpoint-*') if p.name.split('-')[-1].isdigit()],
        key=lambda p: int(p.name.split('-')[-1]),
    )
    resume_from = str(checkpoints[-1]) if checkpoints else None
    if resume_from:
        print(f'Resuming from checkpoint: {resume_from}')

    trainer = GRPOTrainer(
        model=agents['lead'].model,
        processing_class=tokenizer,
        reward_funcs=[lead_reward_fn],
        args=config,
        train_dataset=ds_lead,
        peft_config=None,
        formatting_func=lambda ex: ex['prompt'],
    )

    trainer.train(resume_from_checkpoint=resume_from)
    agents['lead'].model.save_pretrained('artifacts/lead_adapter')
    tokenizer.save_pretrained('artifacts/lead_adapter')
    print('✓ Lead adapter trained & saved to artifacts/lead_adapter')

In [ ]:
# Cell C — Save and push Lead adapter to HF Hub
from huggingface_hub import HfApi

model.save_pretrained('checkpoints/lead_adapter_v1', selected_adapters=['lead_adapter'])
HfApi().upload_folder(
    folder_path='checkpoints/lead_adapter_v1',
    repo_id=f'{HF_USER}/sdk-sovereign-lead-adapter',
    repo_type='model',
    commit_message='Lead LoRA adapter v1 (GRPO round 1)',
)
print(f'✓ Lead adapter pushed to HF Hub: {HF_USER}/sdk-sovereign-lead-adapter')